In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

print(f"Project path is: {project_root}")

# Add project root directory to Python path so local modules (src/) can be imported
sys.path.append(str(project_root))

Project path is: /Users/jeet/Documents/Development/Projects/AI-Engineering-Portfolio/pdf-rag-assistant


# 1. PDF Extraction

In [2]:
from src.document_loader import get_pdf_metadata, load_pdf

/Users/jeet/Documents/Development/Projects/AI-Engineering-Portfolio/pdf-rag-assistant/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
pdf_path = './../data/pdfs/Bert_Model.pdf'

documents = load_pdf(pdf_path)
print(f"Total pages: {len(documents)}")

Total pages: 16


### Check first page:

In [4]:
first_page = documents[0]

print(first_page.page_content[:500])
print(first_page.metadata)

Proceedings of NAACL-HLT 2019, pages 4171–4186
Minneapolis, Minnesota, June 2 - June 7, 2019. c⃝2019 Association for Computational Linguistics
4171
BERT: Pre-training of Deep Bidirectional Transformers for
Language Understanding
Jacob Devlin Ming-Wei Chang Kenton Lee Kristina Toutanova
Google AI Language
{jacobdevlin,mingweichang,kentonl,kristout}@google.com
Abstract
We introduce a new language representa-
tion model called BERT, which stands for
Bidirectional Encoder Representations from
Transf
{'producer': 'pdfTeX-1.40.18', 'creator': 'LaTeX with hyperref package', 'creationdate': '2019-04-29T17:36:03+00:00', 'author': 'Jacob Devlin ; Ming-Wei Chang ; Kenton Lee ; Kristina Toutanova', 'keywords': '', 'moddate': '2019-04-29T17:36:03+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.18 (TeX Live 2017) kpathsea version 6.2.3', 'subject': 'N19-1 2019', 'title': 'BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding', 'trapped': '/False', '

### Get metadata:

In [5]:
metadata = get_pdf_metadata(documents)
metadata

{'total_page': 16, 'source': '../data/pdfs/Bert_Model.pdf'}

# 2. Text Chunking

In [6]:
from src.chunking import split_document

In [7]:
chunks = split_document(
	documents,
	chunk_size=1000,
	chunk_overlap=100
)

print(f"Total chunks: {len(chunks)}")

Total chunks: 78


In [8]:
print("First 3 chunks are:\n")
for chunk in chunks[:3]:
	print(f"Chunk {chunk.metadata["chunk_id"]} is:")
	print(chunk)
	print("-" * 50)

First 3 chunks are:

Chunk 1 is:
page_content='Proceedings of NAACL-HLT 2019, pages 4171–4186
Minneapolis, Minnesota, June 2 - June 7, 2019. c⃝2019 Association for Computational Linguistics
4171
BERT: Pre-training of Deep Bidirectional Transformers for
Language Understanding
Jacob Devlin Ming-Wei Chang Kenton Lee Kristina Toutanova
Google AI Language
{jacobdevlin,mingweichang,kentonl,kristout}@google.com
Abstract
We introduce a new language representa-
tion model called BERT, which stands for
Bidirectional Encoder Representations from
Transformers. Unlike recent language repre-
sentation models (Peters et al., 2018a; Rad-
ford et al., 2018), BERT is designed to pre-
train deep bidirectional representations from
unlabeled text by jointly conditioning on both
left and right context in all layers. As a re-
sult, the pre-trained BERT model can be ﬁne-
tuned with just one additional output layer
to create state-of-the-art models for a wide
range of tasks, such as question answering and
lang

# 3. Embeddings

In [9]:
from src.embeddings import generate_embeddings, get_embedding_model

In [10]:
embedding_model = get_embedding_model()

# No need to create embeddings here, as we will do this in create_vector_store from vector_store.py
embeddings = generate_embeddings(
	documents=chunks,
	embedding_model=embedding_model
)

Using device: mps


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9226.20it/s]


In [11]:
print(f"Total embeddings: {len(embeddings)}")
print(f"Length of each embedding: {len(embeddings[0])}")

Total embeddings: 78
Length of each embedding: 384


# 4. Vector Store

In [12]:
from src.vector_store import (
	create_vector_store,
	load_vector_store,
	save_vector_store,
)

### Create FAISS Vector Store

In [13]:
vector_store = create_vector_store(
	documents=chunks,
	embedding_model=embedding_model
)

print("FAISS vector store created successfully.")

FAISS vector store created successfully.


### Save FAISS Index

In [14]:
save_vector_store(vector_store)

print("FAISS index saved successfully.")

FAISS index saved successfully.


### Verify Files

In [15]:
from pathlib import Path

index_path = Path("./../vectorstore/faiss_index")

for file in index_path.iterdir():
	print(file.name)

index.faiss
index.pkl


### Load FAISS Index

In [16]:
loaded_vector_store = load_vector_store(
	embedding_model
)

print("FAISS index loaded successfully.")

FAISS index loaded successfully.


### Verify Index Size

In [17]:
loaded_vector_store.index.ntotal

78

### Verify Stored Documents

In [18]:
len(loaded_vector_store.docstore._dict)

78

### Summary

In [19]:
print("=" * 50)
print("FAISS Integration Successful")
print("=" * 50)

print(f"Pages Loaded      : {len(documents)}")
print(f"Chunks Created    : {len(chunks)}")
print(f"Embeddings        : {len(embeddings)}")
print(f"Vectors in FAISS  : {loaded_vector_store.index.ntotal}")

FAISS Integration Successful
Pages Loaded      : 16
Chunks Created    : 78
Embeddings        : 78
Vectors in FAISS  : 78


# 5. Retrieval

In [ ]:
from src.retrieval import create_retriever, retrive_documents

### Create Retriever

In [ ]:
retriever = create_retriever(
	vector_store=loaded_vector_store,
	k=4,
)

print("Retriever created successfully.")

Retriever created successfully.


### Ask a Question

In [ ]:
question = "What architecture does BERT use?"

documents = retrive_documents(
	question=question,
	retriever=retriever
)

print(f"Retrieved {len(documents)} documents.")

Retrieved 4 documents.


### Display Retrieved Chunks

In [23]:
for index, document in enumerate(documents, start=1):
	print("=" * 80)
	print(f"Result {index}")
	print("=" * 80)

	print(f"Page 		: {document.metadata.get('page')}")
	print(f"Chunk ID	: {document.metadata.get('chunk_id')}")
	print(f"Source		: {document.metadata.get('source')}")

	print("\nContent:\n")
	print(document.page_content[:200])
	print()

Result 1
Page 		: 1
Chunk ID	: 7
Source		: ../data/pdfs/Bert_Model.pdf

Content:

the need for many heavily-engineered task-
speciﬁc architectures. BERT is the ﬁrst ﬁne-
tuning based representation model that achieves
state-of-the-art performance on a large suite
of sentence-level 

Result 2
Page 		: 2
Chunk ID	: 13
Source		: ../data/pdfs/Bert_Model.pdf

Content:

tuning, the BERT model is ﬁrst initialized with
the pre-trained parameters, and all of the param-
eters are ﬁne-tuned using labeled data from the
downstream tasks. Each downstream task has sep-
arate 

Result 3
Page 		: 2
Chunk ID	: 11
Source		: ../data/pdfs/Bert_Model.pdf

Content:

4173
BERT BERT
E[CLS] E1  E[SEP]... EN E1’ ... EM’
C
 T1
 T[SEP]...
 TN
 T1’ ...
 TM’
[CLS] Tok 1  [SEP]... Tok N Tok 1 ... TokM
Question Paragraph
Start/End Span
BERT
E[CLS] E1  E[SEP]... EN E1’ ... 

Result 4
Page 		: 12
Chunk ID	: 62
Source		: ../data/pdfs/Bert_Model.pdf

Content:

4183
BERT (Ours)
Trm Trm Trm
Trm Trm Trm
...
...
Trm Trm Trm
T

### Test Another Question

In [ ]:
question = "What is masked language modeling?"

documents = retrive_documents(
	question=question,
	retriever=retriever
)

print(f"Retrieved {len(documents)} documents.")

Retrieved 4 documents.


### Display Results

In [25]:
for index, document in enumerate(documents, start=1):
	print("=" * 80)
	print(f"Result {index}")
	print("=" * 80)

	print(f"Page		: {document.metadata.get('page')}")
	print(f"Chunk ID	: {document.metadata.get('chunk_id')}")

	print("\nPreview:\n")
	print(document.page_content[:200])
	print()

Result 1
Page		: 3
Chunk ID	: 19

Preview:

case, the ﬁnal hidden vectors corresponding to the
mask tokens are fed into an output softmax over
the vocabulary, as in a standard LM. In all of our
experiments, we mask 15% of all WordPiece to-
kens

Result 2
Page		: 3
Chunk ID	: 18

Preview:

strictly more powerful than either a left-to-right
model or the shallow concatenation of a left-to-
right and a right-to-left model. Unfortunately,
standard conditional language models can only be
tra

Result 3
Page		: 0
Chunk ID	: 5

Preview:

as question answering, where it is crucial to incor-
porate context from both directions.
In this paper, we improve the ﬁne-tuning based
approaches by proposing BERT: Bidirectional
Encoder Representat

Result 4
Page		: 15
Chunk ID	: 76

Preview:

In Section 3.1, we mention that BERT uses a
mixed strategy for masking the target tokens when
pre-training with the masked language model
(MLM) objective. The following is an ablation
study to evaluat



### Test Empty Question

In [ ]:
try:
    retrive_documents(
        question="",
        retriever=retriever,
    )

except Exception as error:  # noqa: BLE001
    print(error)

Question cannot be empty.


### Change value of k

In [ ]:
retriever = create_retriever(
	vector_store=vector_store,
	k=2,
)

documents = retrive_documents(
	question="What architecture does BERT use?",
	retriever=retriever
)

print(f"Retrieved {len(documents)} documents.")

Retrieved 2 documents.


### Verify Retrieved Metadata

In [28]:
for document in documents:
	print(document.metadata)

{'producer': 'pdfTeX-1.40.18', 'creator': 'LaTeX with hyperref package', 'creationdate': '2019-04-29T17:36:03+00:00', 'author': 'Jacob Devlin ; Ming-Wei Chang ; Kenton Lee ; Kristina Toutanova', 'keywords': '', 'moddate': '2019-04-29T17:36:03+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.18 (TeX Live 2017) kpathsea version 6.2.3', 'subject': 'N19-1 2019', 'title': 'BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding', 'trapped': '/False', 'source': '../data/pdfs/Bert_Model.pdf', 'total_pages': 16, 'page': 1, 'page_label': '2', 'chunk_id': 7}
{'producer': 'pdfTeX-1.40.18', 'creator': 'LaTeX with hyperref package', 'creationdate': '2019-04-29T17:36:03+00:00', 'author': 'Jacob Devlin ; Ming-Wei Chang ; Kenton Lee ; Kristina Toutanova', 'keywords': '', 'moddate': '2019-04-29T17:36:03+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.18 (TeX Live 2017) kpathsea version 6.2.3', 'subject': 'N19-1 2019', 'title': 'BER

### Summary

In [30]:
print("=" * 50)
print("Retrieval Test Completed")
print("=" * 50)

print(f"Question		: {question}")
print(f"Retrieved Chunks	: {len(documents)}")
print("Top-k Configuration	: 2")

Retrieval Test Completed
Question		: What is masked language modeling?
Retrieved Chunks	: 2
Top-k Configuration	: 2
